# Chapter 15 — Building Systems That Distrust Their Models

**Book alignment:** Hallucination From First Principles, Chapter 15

**Question this notebook isolates:** Does the full path from proposal through enforcement to recording hold when assertion, action, and persistence gates are assembled on one synthetic candidate?

Synthetic fixtures with a perfect structured oracle demonstrate control invariants, not real verifier accuracy.


In [ ]:
from pathlib import Path
import sys
from dataclasses import replace

import numpy as np


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "experiments" / "hallucination-from-first-principles").exists():
            return candidate
    raise RuntimeError(
        "Run this notebook from a checkout containing experiments/hallucination-from-first-principles"
    )


REPO_ROOT = find_repo_root(Path.cwd().resolve())
EXP_ROOT = REPO_ROOT / "experiments" / "hallucination-from-first-principles"
sys.path.insert(0, str(EXP_ROOT))

import recovery_demo as rec
from recovery_demo import RecoveryController, V
import policy_engine as pe
from policy_engine import Context, History, evaluate_policy

rng = np.random.default_rng(15)

FACTUAL = "FACTUAL_EVIDENCE"


class AuthorizationError(Exception):
    pass


def enforce(action, decision, approvals):
    if decision.get("authorization") != "PERMIT":
        raise AuthorizationError(
            f"action {action} requires {decision.get('requirement')}; execution blocked"
        )
    if decision.get("requires_human_approval") and action not in approvals:
        raise AuthorizationError(
            f"action {action} missing human approval; execution blocked"
        )
    return f"EXECUTED:{action}"


def support_energy(states):
    supported = sum(1 for v in states.values() if v == V.SUPPORTED)
    return 1.0 - supported / max(1, len(states))


def admit_fact(*, verified, n_families, has_valid_time):
    if verified and n_families >= 2 and has_valid_time:
        return {"admission": "ADMIT", "lifecycle": "ACTIVE",
                "allowed_uses": ["CONVERSATIONAL_CONTEXT", FACTUAL],
                "matched_rule": "CORROBORATED_TIME_BOUNDED_FACT"}
    return {"admission": "QUARANTINE", "lifecycle": "QUARANTINED",
            "allowed_uses": ["CONVERSATIONAL_CONTEXT", "REGRESSION_TEST"],
            "matched_rule": "ANSWERABLE_BUT_NOT_DURABLE_FACT",
            "missing": [m for m, ok in
                        (("SECOND_INDEPENDENT_SOURCE_FAMILY", n_families >= 2),
                         ("VALID_TIME", has_valid_time)) if not ok]}


## One held case traced end to end

The model proposes `Q3 revenue was approximately $46 million` against evidence holding only Q1/Q2. The energy sensor reports unsupported mass, assertion policy holds for retrieval, the board-send action is held for human approval and its enforcement raises, and the same content is quarantined at the persistence gate. Three boundaries, one case, no invisible authority.


In [ ]:
evidence_v1 = frozenset({"q1_revenue", "q2_revenue"})
cand_v1 = rec.base_candidate()
states_v1 = rec.verify(cand_v1, evidence_v1)
energy_v1 = support_energy(states_v1)

record_v1 = {"containment": "PASS", "relation_fidelity": "PASS",
               "epistemic_adequacy": "RECOVERABLE_EVIDENCE_GAP",
               "provenance": "UNVERIFIED"}
decision_v1 = evaluate_policy(record_v1, Context(), History())
action_v1 = {"authorization": "HOLD", "requirement": "HUMAN_APPROVAL_REQUIRED"}
try:
    enforce("SEND_BOARD_UPDATE", action_v1, approvals=set())
    send_v1 = "EXECUTED (unexpected)"
except AuthorizationError as exc:
    send_v1 = f"BLOCKED: {exc}"
admission_v1 = {"admission": "QUARANTINE", "lifecycle": "QUARANTINED",
                "allowed_uses": ["DEBUG", "REGRESSION_TEST"],
                "matched_rule": "UNVERIFIED_CANDIDATE"}

print("energy:", round(energy_v1, 4))
print("assertion:", decision_v1["commitment"], "/", decision_v1["next_action"],
      "matched=", decision_v1["matched_rule"])
print("action:", send_v1)
print("memory:", admission_v1["admission"], admission_v1["lifecycle"],
      admission_v1["allowed_uses"])


In [ ]:
assert abs(energy_v1 - 2 / 3) < 1e-12
assert (decision_v1["commitment"], decision_v1["next_action"]) == ("HOLD", "RETRIEVE")
assert send_v1.startswith("BLOCKED")
assert admission_v1["admission"] == "QUARANTINE"
assert FACTUAL not in admission_v1["allowed_uses"]

print("held case: measured, held, blocked, quarantined")


## Verified for assertion, still quarantined for persistence

Retrieval adds the authoritative Q3 filing, so a new candidate stating `$47.3M` verifies fully and assertion policy permits it. But durability is a separate gate: one source family and no validity window keeps it quarantined as conversational context. A second independent family plus a recorded quarter promotes the same number to durable factual evidence. Policy replay shows the provenance rule is what changed between versions.


In [ ]:
evidence_v2 = frozenset({"q1_revenue", "q2_revenue", "second_half_outlook", "q3_revenue_value"})
cand_v2 = rec.Candidate(
    id="cand_v2", parent="cand_v1",
    claims=[rec.Claim("c1", "Q2 revenue was $43.8 million.", "q2_revenue"),
            rec.Claim("c2", "Second-half demand is expected to improve.",
                        "second_half_outlook"),
            rec.Claim("c3", "Q3 revenue was $47.3 million.", "q3_revenue_value")])
states_v2 = rec.verify(cand_v2, evidence_v2)
energy_v2 = support_energy(states_v2)
record_v2 = {"containment": "PASS", "relation_fidelity": "PASS",
               "epistemic_adequacy": "ANSWERABLE", "provenance": "VERIFIED"}
decision_v2 = evaluate_policy(record_v2, Context(), History())
single_source = admit_fact(verified=True, n_families=1, has_valid_time=False)
corroborated = admit_fact(verified=True, n_families=2, has_valid_time=True)

replay_rec = {**record_v2, "provenance": "UNVERIFIED"}
replay_v1 = evaluate_policy(replay_rec, Context(), History(),
                             require_provenance=False)
replay_v2 = evaluate_policy(replay_rec, Context(), History(),
                             require_provenance=True)

print("energy:", round(energy_v2, 4))
print("assertion:", decision_v2["commitment"], "/", decision_v2["next_action"],
      "matched=", decision_v2["matched_rule"])
print("memory (one family, no valid time):", single_source["admission"],
      single_source["missing"])
print("memory (two families + quarter):", corroborated["admission"],
      corroborated["lifecycle"], corroborated["allowed_uses"])
print("replay unverified provenance | v1:", replay_v1["commitment"], "/",
      replay_v1["next_action"], "| v2:", replay_v2["commitment"], "/",
      replay_v2["next_action"])


In [ ]:
assert energy_v2 == 0.0
assert all(v == V.SUPPORTED for v in states_v2.values())
assert (decision_v2["commitment"], decision_v2["next_action"]) == ("PERMIT", "NONE")
assert single_source["admission"] == "QUARANTINE"
assert FACTUAL not in single_source["allowed_uses"]
assert corroborated["admission"] == "ADMIT"
assert FACTUAL in corroborated["allowed_uses"]
assert (replay_v1["commitment"], replay_v1["next_action"]) == ("PERMIT", "NONE")
assert (replay_v2["commitment"], replay_v2["next_action"]) == ("HOLD", "VERIFY")

print("PERMIT to state is not ADMIT to durable memory")


## Three boundaries, three planes, one ledger

Proposal (generator), observation (sensor + verifier), and control (policy + enforcement + admission) meet in a replayable ledger. The recovery loop closes the original gap by omission, and counterfactual replay re-evaluates the stored record under a stricter policy in dry-run mode: decisions change, but no email sends and no memory writes execute.


In [ ]:
terminal_fix, final_fix = RecoveryController().run(rec.base_candidate(), "omission")
print("recovery:", terminal_fix, final_fix.id, "parent=", final_fix.parent)

ledger = {"candidate": "cand_v1", "evidence": "evidence_v1",
          "record": record_v1, "policy_version": "policy-v2"}
side_effects = []
replay = evaluate_policy(dict(ledger["record"]), Context(), History())
dry_run = True
replayed_send = "SIMULATED_BLOCK" if dry_run else enforce(
    "SEND_BOARD_UPDATE", action_v1, approvals=set())
print("replay:", replay["commitment"], "/", replay["next_action"],
      "matched=", replay["matched_rule"])
print("replayed send:", replayed_send, "| side effects:", side_effects)

boundaries = {"assertion": decision_v1["commitment"],
              "action": action_v1["authorization"],
              "persistence": admission_v1["admission"]}
print("boundaries:", boundaries)


In [ ]:
assert terminal_fix == "PERMIT" and final_fix.authorized is True
assert (replay["commitment"], replay["next_action"]) == ("HOLD", "RETRIEVE")
assert side_effects == []
assert boundaries == {"assertion": "HOLD", "action": "HOLD",
                        "persistence": "QUARANTINE"}
assert decision_v2["commitment"] == "PERMIT"  # same content class, other gate differs

print("proposal observed, transition authorized, replay dry, ledger complete")


## What we earned

The book's arc closes here. Generation is not acceptance; sensors are not verdicts; distinctions survive until after the decision; different failures route differently; repairs become new states; relevance is not admissibility; transformation manufactures no evidence; side effects need external mediation; persistence needs its own gate; and every reliability claim stays scoped to workload, policy, measurements, evidence, and enforcement.

The reliable unit is not the model but the system around it: the model proposes, the system observes, measures, decides, enforces, and records — earning bounded reliability under explicit assumptions while the model remains fallible.
